In [1]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/mlp/mlp.bit")
ol.ip_dict

{'mlp_axi_v1_0_MLP_AXI_0': {'type': 'xilinx.com:user:mlp_axi_v1_0_MLP_AXI:1.0',
  'mem_id': 'S_AXI',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'C_S_AXI_DATA_WIDTH': '32',
   'C_S_AXI_ADDR_WIDTH': '7',
   'Component_Name': 'design_1_mlp_axi_v1_0_MLP_AXI_0_0',
   'EDK_IPTYPE': 'PERIPHERAL',
   'C_BASEADDR': '0x43C00000',
   'C_HIGHADDR': '0x43C0FFFF',
   'DATA_WIDTH': '32',
   'PROTOCOL': 'AXI4LITE',
   'FREQ_HZ': '50000000',
   'ID_WIDTH': '0',
   'ADDR_WIDTH': '7',
   'AWUSER_WIDTH': '0',
   'ARUSER_WIDTH': '0',
   'WUSER_WIDTH': '0',
   'RUSER_WIDTH': '0',
   'BUSER_WIDTH': '0',
   'READ_WRITE_MODE': 'READ_WRITE',
   'HAS_BURST': '0',
   'HAS_LOCK': '0',
   'HAS_PROT': '1',
   'HAS_CACHE': '0',
   'HAS_QOS': '0',
   'HAS_REGION': '0',
   'HAS_WSTRB': '1',
   'HAS_BRESP': '1',
   'HAS_RRESP': '1',
   'SUPPORTS_NARROW_BURST': '0',
   'NUM_READ_OUTSTANDING': '1',
   'NUM_WRITE_OUTSTANDING': '1',
   'MAX_BURST_LENGTH': '1',
   'PHASE': '0.000',
   'CLK_DO

In [2]:
mlp = ol.mlp_axi_v1_0_MLP_AXI_0
print(hex(mlp.read(0x00)))

0x0


In [3]:
mlp.write(0x04, 0x11223344)
print(hex(mlp.read(0x04)))

0x11223344


In [5]:
 
from pynq import Overlay
import time
import numpy as np
# -----------------------------
# Load overlay
# -----------------------------
ol = Overlay("/home/xilinx/jupyter_notebooks/mlp/mlp.bit")
mlp = ol.mlp_axi_v1_0_MLP_AXI_0
CLASS_NAMES = ["circle", "rectangle", "triangle", "line", "freehand"]
# -----------------------------
# AXI helper functions
# -----------------------------
def pack_features_to_regs(features):
    """
    features: list of 70 signed 8-bit ints (-128..127)
    returns: list of 18 packed 32-bit words
    """
    if len(features) != 70:
        raise ValueError(f"Expected 70 features, got {len(features)}")
    byte_vals = [(x + 256) % 256 for x in features]
    regs = []
    for i in range(0, 70, 4):
        chunk = byte_vals[i:i+4]
        while len(chunk) < 4:
            chunk.append(0)
        word = (
            (chunk[0] << 0)  |
            (chunk[1] << 8)  |
            (chunk[2] << 16) |
            (chunk[3] << 24)
        )
        regs.append(word)
    while len(regs) < 18:
        regs.append(0)
    return regs
def mlp_write_features(features):
    regs = pack_features_to_regs(features)
    for i, word in enumerate(regs):
        addr = 0x04 + 4 * i   # reg1 starts at 0x04
        mlp.write(addr, word)
def mlp_start():
    mlp.write(0x00, 0x1)
    time.sleep(0.01)
    mlp.write(0x00, 0x0)
def mlp_status():
    return mlp.read(0x00)
def mlp_done():
    return (mlp_status() >> 8) & 0x1
def mlp_class_id():
    return (mlp_status() >> 9) & 0x7
def mlp_predict(features, timeout_s=1.0):
    mlp_write_features(features)
    mlp_start()
    t0 = time.time()
    while mlp_done() == 0:
        if time.time() - t0 > timeout_s:
            raise TimeoutError("Timed out waiting for accelerator")
    cid = mlp_class_id()
    label = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else f"unknown({cid})"
    return cid, label, mlp_status()
# -----------------------------
# Preprocessing functions
# -----------------------------
def resample_stroke(stroke, n_points=35):
    stroke = np.array(stroke, dtype=float)
    d = np.sqrt(((stroke[1:] - stroke[:-1]) ** 2).sum(axis=1))
    d = np.insert(d, 0, 0.0)
    cumdist = np.cumsum(d)
    total = cumdist[-1]
    target = np.linspace(0, total, n_points)
    resampled = []
    for t in target:
        idx = np.searchsorted(cumdist, t)
        if idx == 0:
            resampled.append(stroke[0])
        else:
            if idx >= len(stroke):
                idx = len(stroke) - 1
            t1, t2 = cumdist[idx - 1], cumdist[idx]
            p1, p2 = stroke[idx - 1], stroke[idx]
            if t2 == t1:
                resampled.append(p1)
            else:
                alpha = (t - t1) / (t2 - t1)
                resampled.append(p1 + alpha * (p2 - p1))
    return np.array(resampled)
def stroke_to_features(stroke):
    pts = resample_stroke(stroke, 35)
    min_xy = pts.min(axis=0)
    max_xy = pts.max(axis=0)
    pts = (pts - min_xy) / (max_xy - min_xy + 1e-6)
    features = pts.flatten()
    features = (features * 127).astype(np.int8)
    return features.tolist()
# -----------------------------
# Example stroke (triangle)
# -----------------------------
stroke = [
    [1039,459],[1041,463],[1041,470],[1040,482],[1039,495],[1038,510],
    [1037,528],[1037,549],[1039,571],[1042,591],[1045,604],[1045,616],
    [1043,628],[1044,638],[1046,646],[1053,652],[1064,651],[1077,643],
    [1091,631],[1108,616],[1128,598],[1147,587],[1168,578],[1190,571],
    [1209,569],[1223,569],[1235,570],[1241,569],[1245,568],[1250,569],
    [1255,569],[1258,570],[1255,568],[1246,560],[1232,545],[1211,531],
    [1185,519],[1156,506],[1127,497],[1104,486],[1084,477],[1064,468],
    [1047,455],[1029,440],[1014,427],[1005,417]
]
# -----------------------------
# Run end-to-end
# -----------------------------
features = stroke_to_features(stroke)
print("Feature length:", len(features))
print("First 10 features:", features[:10])
print("Status before start:", hex(mlp_status()))
cid, label, status = mlp_predict(features)
print("Predicted class:", label)
print("class_id:", cid)
print("status:", hex(status))
 

Feature length: 70
First 10 features: [17, 23, 17, 34, 16, 46, 16, 57, 16, 69]
Status before start: 0x0


TimeoutError: Timed out waiting for accelerator

In [6]:
print("before:", hex(mlp.read(0x00)))
mlp.write(0x00, 0x1)
print("after write 1:", hex(mlp.read(0x00)))
time.sleep(0.01)
mlp.write(0x00, 0x0)
print("after write 0:", hex(mlp.read(0x00)))

before: 0x800
after write 1: 0x901
after write 0: 0x800


In [7]:
 
from pynq import Overlay
import time
import numpy as np
# -----------------------------
# Load overlay
# -----------------------------
ol = Overlay("/home/xilinx/jupyter_notebooks/mlp/mlp.bit")
mlp = ol.mlp_axi_v1_0_MLP_AXI_0
CLASS_NAMES = ["circle", "rectangle", "triangle", "line", "freehand"]
# -----------------------------
# AXI helper functions
# -----------------------------
def pack_features_to_regs(features):
    """
    features: list of 70 signed 8-bit ints (-128..127)
    returns: list of 18 packed 32-bit words
    """
    if len(features) != 70:
        raise ValueError(f"Expected 70 features, got {len(features)}")
    byte_vals = [(x + 256) % 256 for x in features]
    regs = []
    for i in range(0, 70, 4):
        chunk = byte_vals[i:i+4]
        while len(chunk) < 4:
            chunk.append(0)
        word = (
            (chunk[0] << 0)  |
            (chunk[1] << 8)  |
            (chunk[2] << 16) |
            (chunk[3] << 24)
        )
        regs.append(word)
    while len(regs) < 18:
        regs.append(0)
    return regs
def mlp_write_features(features):
    regs = pack_features_to_regs(features)
    for i, word in enumerate(regs):
        addr = 0x04 + 4 * i   # reg1 starts at 0x04
        mlp.write(addr, word)
def mlp_start_hold():
    mlp.write(0x00, 0x1)
def mlp_stop():
    mlp.write(0x00, 0x0)
def mlp_status():
    return mlp.read(0x00)
def mlp_done():
    return (mlp_status() >> 8) & 0x1
def mlp_class_id():
    return (mlp_status() >> 9) & 0x7
def mlp_predict(features, timeout_s=1.0):
    mlp_write_features(features)
    mlp_start_hold()
    t0 = time.time()
    while mlp_done() == 0:
        if time.time() - t0 > timeout_s:
            mlp_stop()
            raise TimeoutError("Timed out waiting for accelerator")
    status = mlp_status()
    cid = (status >> 9) & 0x7
    label = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else f"unknown({cid})"
    mlp_stop()
    return cid, label, status
# -----------------------------
# Preprocessing functions
# -----------------------------
def resample_stroke(stroke, n_points=35):
    stroke = np.array(stroke, dtype=float)
    d = np.sqrt(((stroke[1:] - stroke[:-1]) ** 2).sum(axis=1))
    d = np.insert(d, 0, 0.0)
    cumdist = np.cumsum(d)
    total = cumdist[-1]
    target = np.linspace(0, total, n_points)
    resampled = []
    for t in target:
        idx = np.searchsorted(cumdist, t)
        if idx == 0:
            resampled.append(stroke[0])
        else:
            if idx >= len(stroke):
                idx = len(stroke) - 1
            t1, t2 = cumdist[idx - 1], cumdist[idx]
            p1, p2 = stroke[idx - 1], stroke[idx]
            if t2 == t1:
                resampled.append(p1)
            else:
                alpha = (t - t1) / (t2 - t1)
                resampled.append(p1 + alpha * (p2 - p1))
    return np.array(resampled)
def stroke_to_features(stroke):
    pts = resample_stroke(stroke, 35)
    min_xy = pts.min(axis=0)
    max_xy = pts.max(axis=0)
    pts = (pts - min_xy) / (max_xy - min_xy + 1e-6)
    features = pts.flatten()
    features = (features * 127).astype(np.int8)
    return features.tolist()
# -----------------------------
# Example stroke (triangle)
# -----------------------------
stroke = [
    [1039,459],[1041,463],[1041,470],[1040,482],[1039,495],[1038,510],
    [1037,528],[1037,549],[1039,571],[1042,591],[1045,604],[1045,616],
    [1043,628],[1044,638],[1046,646],[1053,652],[1064,651],[1077,643],
    [1091,631],[1108,616],[1128,598],[1147,587],[1168,578],[1190,571],
    [1209,569],[1223,569],[1235,570],[1241,569],[1245,568],[1250,569],
    [1255,569],[1258,570],[1255,568],[1246,560],[1232,545],[1211,531],
    [1185,519],[1156,506],[1127,497],[1104,486],[1084,477],[1064,468],
    [1047,455],[1029,440],[1014,427],[1005,417]
]
# -----------------------------
# Run end-to-end
# -----------------------------
features = stroke_to_features(stroke)
print("Feature length:", len(features))
print("First 10 features:", features[:10])
print("Status before start:", hex(mlp_status()))
cid, label, status = mlp_predict(features, timeout_s=1.0)
print("Predicted class:", label)
print("class_id:", cid)
print("status:", hex(status))
 

Feature length: 70
First 10 features: [17, 23, 17, 34, 16, 46, 16, 57, 16, 69]
Status before start: 0x0
Predicted class: freehand
class_id: 4
status: 0x901


In [21]:
 
import json
with open("/home/xilinx/jupyter_notebooks/mlp/triangle_features.json") as f:
    data = json.load(f)
features = data["features"]
print("Feature length:", len(features))
print("First 10 features:", features[:10])
 
 

Feature length: 70
First 10 features: [0.5, -0.3119460555083364, 0.43290423395953864, -0.26914358406873173, 0.36330117225489256, -0.23213023143200553, 0.2861814128707007, -0.2124722535497605, 0.20687198457345293, -0.21363069029543877]


In [22]:
 
 
import numpy as np
features = np.array(data["features"])
# scale coordinates normally
features[:64] = np.round(features[:64] * 127)
# scale geometry features more gently
features[64:] = np.round(features[64:] * 32)
features = features.astype(np.int8)
features = features.tolist()
print(features[:10])
print(features[64:])
 

[64, -40, 55, -34, 46, -29, 36, -27, 26, -27]
[0, 15, 51, 79, 4, -8]


In [23]:
cid, label, status = mlp_predict(features)

print("Predicted class:", label)
print("class_id:", cid)
print("status:", hex(status))

Predicted class: circle
class_id: 0
status: 0x101


In [18]:
f=np.array(data["features"])
print("min:", f.min())
print("max:", f.max())

min: -0.5
max: 2.708834701644498


In [24]:
 
import json
with open("/home/xilinx/jupyter_notebooks/mlp/rectangle_features.json") as f:
    data = json.load(f)
features = data["features"]
print("Feature length:", len(features))
print("First 10 features:", features[:10])
features = np.array(data["features"])
# scale coordinates normally
features[:64] = np.round(features[:64] * 127)
# scale geometry features more gently
features[64:] = np.round(features[64:] * 32)
features = features.astype(np.int8)
features = features.tolist()
print(features[:10])
print(features[64:])

Feature length: 70
First 10 features: [-0.4209621993127148, -0.4006301844587671, -0.4965635738831615, -0.35252021882302786, -0.4202068421672519, -0.28303167447051325, -0.3128255164438908, -0.26796750043450857, -0.20426647783826468, -0.2643919041827473]
[-53, -51, -63, -45, -53, -36, -40, -34, -26, -34]
[32, 3, 40, 108, 3, 20]


In [25]:
cid, label, status = mlp_predict(features)

print("Predicted class:", label)
print("class_id:", cid)
print("status:", hex(status))

Predicted class: rectangle
class_id: 1
status: 0x301
